# AlphaDent YOLO26 Segmentation Pipeline (End-to-End)

This notebook contains the complete pipeline for downloading the AlphaDent dataset, training, validating, and performing inference using **YOLO26 segment models**.

### Models Supported:
- `yolo26n-seg.pt` (Nano)
- `yolo26s-seg.pt` (Small)
- `yolo26m-seg.pt` (Medium)
- `yolo26l-seg.pt` (Large)
- `yolo26x-seg.pt` (Extra Large)

Developed for running on **Kaggle**.

## 1. Clone Repository & Setup Working Directory

In [ ]:
# Clone repo if not running inside it already
import os
if not os.path.exists('/kaggle/working/AlphaDentYolov26'):
    print("Cloning AlphaDentYolov26 repository...")
    !git clone https://github.com/ShMazumder/AlphaDentYolov26 /kaggle/working/AlphaDentYolov26/
else:
    print("Repository already exists.")

# Change directory to the repository root
os.chdir('/kaggle/working/AlphaDentYolov26/')
print(f"Current working directory: {os.getcwd()}")

# Install dependencies
print("Installing dependencies...")
!pip install -q -r requirements.txt
!pip install -q matplotlib opencv-python

## 2. Configuration

Configure model selection and training hyperparameters here.

In [ ]:
import torch

# Experiment Grid Configuration
GRID_MODELS = ['yolo26n-seg.pt', 'yolo26s-seg.pt', 'yolo26m-seg.pt', 'yolo26l-seg.pt', 'yolo26x-seg.pt']
GRID_IMAGE_SIZES = [640, 960]
TARGET_BATCH_SIZE = 16  # Initial target batch size; will auto-fallback to 8 or 4 on CUDA OOM
EPOCHS = 100

# Override Checkpoint Controls
# If False, will skip running if results/weights for the configuration already exist on disk
OVERRIDE_TRAIN = False
OVERRIDE_VAL = False
OVERRIDE_XAI = False

# Dataset configuration file path
DATA_CONFIG = './AlphaDent/yolo_seg_train.yaml'

# Device configuration: list of all available GPU IDs, or 'cpu'
DEVICE = list(range(torch.cuda.device_count())) if torch.cuda.is_available() else 'cpu'

# Validation & Inference Settings
CONF_THRESHOLD = 0.001
IOU_THRESHOLD = 0.5

print("--- Experiment Grid Setup ---")
print(f"Models to test: {GRID_MODELS}")
print(f"Resolutions to test: {GRID_IMAGE_SIZES}")
print(f"Device: {DEVICE}")
print(f"Target Batch Size: {TARGET_BATCH_SIZE}, Epochs per run: {EPOCHS}")
print(f"Override controls -> Train: {OVERRIDE_TRAIN}, Val: {OVERRIDE_VAL}, XAI: {OVERRIDE_XAI}")

## 3. Dataset Download & Unpacking

Download the dataset from Zenodo and extract it to the correct path.

In [ ]:
import zipfile
import shutil
import os

input_dir = "/kaggle/input/competitions/alpha-dent/AlphaDent"
working_dir = "/kaggle/working/AlphaDentYolov26/AlphaDent"
zip_path = "/kaggle/working/dataset.zip"

# Check if Kaggle input contains the dataset
if os.path.exists(os.path.join(input_dir, 'images')) and os.listdir(os.path.join(input_dir, 'images')):
    # print("Dataset found in Kaggle input. Preparing files...")
    # os.makedirs(working_dir, exist_ok=True)
    # # 1. Symlink the large images folder (read-only is fine)
    # src_images = os.path.join(input_dir, 'images')
    # dst_images = os.path.join(working_dir, 'images')
    # if not os.path.exists(dst_images):
    #     os.symlink(src_images, dst_images)
    #     print("Symlinked images directory.")
    # # 2. Copy the small labels folder (needs to be writeable for YOLO cache writing)
    # src_labels = os.path.join(input_dir, 'labels')
    # dst_labels = os.path.join(working_dir, 'labels')
    # if not os.path.exists(dst_labels):
    #     print("Copying labels directory to allow cache writing...")
    #     shutil.copytree(src_labels, dst_labels)
    #     print("Copied labels directory.")
    # print("Dataset prepared successfully using Kaggle input.")

    print("Dataset found in Kaggle input. Copying dataset...")
    shutil.copytree(input_dir, working_dir, dirs_exist_ok=True)
    print("Dataset copied successfully. Using input dataset.")
else:
    # Fallback to downloading from Zenodo
    images_dir = os.path.join(working_dir, 'images')
    if not os.path.exists(images_dir) or not os.listdir(images_dir):
        print("Downloading AlphaDent dataset from Zenodo...")
        !wget -q --show-progress "https://zenodo.org/records/16582489/files/AlphaDent.zip?download=1" -O {zip_path}
        
        print("Extracting dataset...")
        os.makedirs(working_dir, exist_ok=True)
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall(working_dir)
        
        # Remove zip to save disk space
        if os.path.exists(zip_path):
            os.remove(zip_path)
            
        # Check if nested folder exists after extraction and move files up if needed
        nested_dir = os.path.join(working_dir, 'AlphaDent')
        if os.path.exists(nested_dir) and os.path.isdir(nested_dir):
            print("Adjusting dataset directory structure...")
            for item in os.listdir(nested_dir):
                shutil.move(os.path.join(nested_dir, item), working_dir)
            os.rmdir(nested_dir)
            
        print("Dataset prepared successfully in working directory.")
    else:
        print("Dataset already exists in working directory.")

## 3.1 Dataset Statistics

Compute and display summary statistics of the dataset images and instance distribution per class.

In [ ]:
import os
import pandas as pd
import yaml
from IPython.display import display, Markdown

working_dir = "./AlphaDent"
train_labels_dir = os.path.join(working_dir, "labels/train")
val_labels_dir = os.path.join(working_dir, "labels/valid")

# Load class configuration dynamically from the dataset YAML file
with open(DATA_CONFIG, 'r') as f:
    data_yaml = yaml.safe_load(f)
class_names = data_yaml['names']

def compute_stats(labels_dir):
    stats = {int(k): 0 for k in class_names.keys()}
    img_count = 0
    if os.path.exists(labels_dir):
        for filename in os.listdir(labels_dir):
            if filename.endswith(".txt"):
                img_count += 1
                filepath = os.path.join(labels_dir, filename)
                with open(filepath, 'r') as f:
                    for line in f:
                        parts = line.strip().split()
                        if len(parts) > 0:
                            class_idx = int(parts[0])
                            if class_idx in stats:
                                stats[class_idx] += 1
    return img_count, stats

train_imgs, train_stats = compute_stats(train_labels_dir)
val_imgs, val_stats = compute_stats(val_labels_dir)

# 1. Image Counts Summary Table
img_df = pd.DataFrame({
    "Split": ["Train", "Validation", "Total"],
    "Images": [train_imgs, val_imgs, train_imgs + val_imgs]
})

# 2. Class Instance Distribution Table
class_rows = []
for idx, name in sorted(class_names.items()):
    t_count = train_stats.get(idx, 0)
    v_count = val_stats.get(idx, 0)
    class_rows.append({
        "Class ID": idx,
        "Class Name": name,
        "Train Instances": t_count,
        "Val Instances": v_count,
        "Total Instances": t_count + v_count
    })
class_df = pd.DataFrame(class_rows)

print("### Dataset Image Summary")
display(Markdown(img_df.to_markdown(index=False)))
print("\n### Dataset Instance Distribution by Class")
display(Markdown(class_df.to_markdown(index=False)))

## 4. Train Model

Load the selected YOLO26 model and start training.

In [ ]:
import os
if os.path.exists('/kaggle/working/AlphaDentYolov26'):
    os.chdir('/kaggle/working/AlphaDentYolov26')

import gc
import torch
from ultralytics import YOLO

# Ensure W&B is disabled to run offline smoothly
os.environ['WANDB_DISABLED'] = 'true'

print("Starting grid search training loop...")

for model_name in GRID_MODELS:
    for img_size in GRID_IMAGE_SIZES:
        model_base = model_name.replace('-seg.pt', '').replace('.pt', '')
        project_name = f'yolo_seg_{model_base}_proj_{img_size}_b{TARGET_BATCH_SIZE}_e{EPOCHS}'
        
        # Check if weights already exist
        possible_paths = [
            f"runs/segment/{project_name}/train/weights/best.pt",
            f"{project_name}/train/weights/best.pt",
            f"runs/{project_name}/train/weights/best.pt",
            f"/kaggle/working/AlphaDentYolov26/runs/segment/{project_name}/train/weights/best.pt"
        ]
        
        weights_exist = False
        existing_weights_path = None
        for p in possible_paths:
            if os.path.exists(p):
                weights_exist = True
                existing_weights_path = p
                break
                
        if weights_exist and not OVERRIDE_TRAIN:
            print(f"Skipping training for {model_name} at size {img_size} (weights exist at {existing_weights_path})")
            continue
            
        print(f"\n==================================================")
        print(f"TRAINING: {model_name} | Image Size: {img_size}")
        print(f"==================================================")
        
        current_batch = TARGET_BATCH_SIZE
        trained_successfully = False
        
        while current_batch >= 4:
            print(f"Attempting training with batch size: {current_batch}...")
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
                
            try:
                model = YOLO(model_name)
                results = model.train(
                    data=DATA_CONFIG,
                    epochs=EPOCHS,
                    imgsz=img_size,
                    batch=current_batch,
                    project=project_name,
                    name="train",
                    exist_ok=True,  # Reuse/overwrite output directory to avoid duplicate folders
                    deterministic=False,
                    plots=True,
                    device=DEVICE,
                )
                trained_successfully = True
                print(f"Successfully trained {model_name} at size {img_size} with batch {current_batch}!")
                break
            except RuntimeError as e:
                if "out of memory" in str(e).lower():
                    print(f"⚠️ CUDA Out of Memory with batch size {current_batch}.")
                    current_batch //= 2
                    if current_batch < 4:
                        print(f"❌ Batch size fell below 4. Skipping training for this config.")
                        break
                    print(f"Retrying with batch size: {current_batch}...")
                else:
                    raise e
                    
print("Training phase finished.")

## 5. Model Validation

Evaluate the trained model's performance on the validation dataset.

In [ ]:
import os
if os.path.exists('/kaggle/working/AlphaDentYolov26'):
    os.chdir('/kaggle/working/AlphaDentYolov26')

import gc
import torch
import pandas as pd
from ultralytics import YOLO

results_file = "grid_search_results.csv"
grid_results = []

# Load previously saved results if file exists
if os.path.exists(results_file):
    try:
        df_existing = pd.read_csv(results_file)
        grid_results = df_existing.to_dict('records')
        print(f"Loaded {len(grid_results)} existing results from {results_file}.")
    except Exception as e:
        print(f"Could not load existing CSV: {e}. Starting fresh.")

print("Starting validation loop...")

for model_name in GRID_MODELS:
    for img_size in GRID_IMAGE_SIZES:
        model_base = model_name.replace('-seg.pt', '').replace('.pt', '')
        project_name = f'yolo_seg_{model_base}_proj_{img_size}_b{TARGET_BATCH_SIZE}_e{EPOCHS}'
        
        # Check if this run is already completed in grid_results
        already_evaluated = False
        for r in grid_results:
            if r.get('model') == model_name and r.get('img_size') == img_size:
                already_evaluated = True
                break
                
        if already_evaluated and not OVERRIDE_VAL:
            print(f"Skipping validation for {model_name} at size {img_size} (metrics exist in CSV)")
            continue
            
        # Check if weights exist before executing validation
        possible_paths = [
            f"runs/segment/{project_name}/train/weights/best.pt",
            f"{project_name}/train/weights/best.pt",
            f"runs/{project_name}/train/weights/best.pt",
            f"/kaggle/working/AlphaDentYolov26/runs/segment/{project_name}/train/weights/best.pt"
        ]
        
        weights_path = None
        for p in possible_paths:
            if os.path.exists(p):
                weights_path = p
                break
                
        if not weights_path:
            print(f"⚠️ Weights not found for {model_name} at size {img_size}. Skipping validation.")
            continue
            
        print(f"\n==================================================")
        print(f"VALIDATING: {model_name} | Image Size: {img_size}")
        print(f"==================================================")
        
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            
        val_device = DEVICE[0] if isinstance(DEVICE, list) else DEVICE
        model_val = YOLO(weights_path)
        
        try:
            metrics = model_val.val(
                data=DATA_CONFIG,
                project=project_name,
                name="val",
                imgsz=img_size,
                batch=1,
                iou=IOU_THRESHOLD,
                conf=CONF_THRESHOLD,
                half=True,
                save_json=False,
                save_txt=False,
                save_conf=False,
                plots=True,
                device=val_device,
            )
        except RuntimeError as e:
            if "out of memory" in str(e).lower() and val_device != "cpu":
                print("⚠️ CUDA Out of Memory during validation. Retrying on CPU...")
                gc.collect()
                torch.cuda.empty_cache()
                metrics = model_val.val(
                    data=DATA_CONFIG,
                    project=project_name,
                    name="val",
                    imgsz=img_size,
                    batch=1,
                    iou=IOU_THRESHOLD,
                    conf=CONF_THRESHOLD,
                    save_json=False,
                    save_txt=False,
                    save_conf=False,
                    plots=True,
                    device="cpu",
                )
            else:
                raise e
                
        # Extract model parameters count dynamically
        params_count = sum(p.numel() for p in model_val.model.parameters())
        
        result_entry = {
            'model': model_name,
            'img_size': img_size,
            'params_M': round(params_count / 1e6, 2),
            'Box_Precision': metrics.box.mp,
            'Box_Recall': metrics.box.mr,
            'Box_mAP50': metrics.box.map50,
            'Box_mAP50_95': metrics.box.map,
            'Mask_Precision': metrics.seg.mp,
            'Mask_Recall': metrics.seg.mr,
            'Mask_mAP50': metrics.seg.map50,
            'Mask_mAP50_95': metrics.seg.map
        }
        
        # Update or append
        existing_idx = -1
        for idx, r in enumerate(grid_results):
            if r.get('model') == model_name and r.get('img_size') == img_size:
                existing_idx = idx
                break
        if existing_idx != -1:
            grid_results[existing_idx] = result_entry
        else:
            grid_results.append(result_entry)
            
        # Save intermediate CSV
        pd.DataFrame(grid_results).to_csv(results_file, index=False)
        print(f"Saved results to {results_file}.")

# Display Final Table
if grid_results:
    df_final = pd.DataFrame(grid_results)
    print("\n" + "="*50)
    print("GRID SEARCH RESULTS SUMMARY TABLE")
    print("="*50)
    try:
        print(df_final.to_markdown(index=False))
    except Exception:
        print(df_final.to_string(index=False))
else:
    print("No validation results found.")

## 6. Inference and Visualization

Perform inference on test images and display the predictions.

In [ ]:
import glob
import cv2
import matplotlib.pyplot as plt

# Run inference on validation/test images to see quality
test_images_dir = "./AlphaDent/images/valid/"
output_dir = "./inference_output/"

model_infer = YOLO(weights_path)

print(f"Running inference on images from: {test_images_dir}")
val_device = DEVICE[0] if isinstance(DEVICE, list) else DEVICE
results_infer = model_infer.predict(
    source=test_images_dir,
    project=output_dir,
    name='preds',
    imgsz=IMAGE_SIZE,
    conf=0.25,  # higher threshold for visual clarity
    iou=0.6,
    half=True,
    save=True,
    save_txt=True,
    device=val_device
)

# Plot the first few results
possible_dirs = [
    f"{output_dir}/preds",
    "runs/segment/inference_output/preds",
    "/kaggle/working/AlphaDentYolov26/runs/segment/inference_output/preds",
    "/kaggle/working/AlphaDentYolov26/inference_output/preds"
]
saved_images = []
for d in possible_dirs:
    saved_images += glob.glob(f"{d}/*.jpg") + glob.glob(f"{d}/*.png")
if saved_images:
    num_display = min(3, len(saved_images))
    fig, axes = plt.subplots(1, num_display, figsize=(18, 6))
    if num_display == 1:
        axes = [axes]
    for idx, img_path in enumerate(saved_images[:num_display]):
        img = cv2.imread(img_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        axes[idx].imshow(img)
        axes[idx].axis('off')
        axes[idx].set_title(os.path.basename(img_path))
    plt.tight_layout()
    plt.show()
else:
    print("No output images found to display.")

## 7. Model Explainability (XAI)

To build trust in medical diagnosis applications, we must explain the model's predictions. This section implements **post-hoc Explainable AI (XAI)** methods to visualize which image regions the YOLO26 model pays attention to when detecting tooth pathologies.

### Supported CAM Methods:
1. **Eigen-CAM**: Computes principal components of the activations. Gradient-free and extremely robust.
2. **Grad-CAM**: Visualizes gradients of the class score with respect to intermediate activations.
3. **Grad-CAM++**: An advanced gradient-based method using second-order gradients.
4. **XGrad-CAM**: Scales the gradients by the normalized activations.
5. **HiRes-CAM**: Element-wise multiplies activations with gradients; guarantees faithfulness.
6. **Layer-CAM**: Spatially weights activations by positive gradients (excellent for lower/intermediate layers).
7. **EigenGrad-CAM**: First principal component of (Activations * Gradients).

### Literature References:
- **Reference Paper (Medical YOLO XAI)**: Borah et al., 2024. ["A comprehensive study on Explainable AI using YOLO and post hoc method on medical diagnosis"](https://doi.org/10.1088/1742-6596/2919/1/012045).
- **Grad-CAM**: Selvaraju et al., 2017. ["Grad-CAM: Visual Explanations from Deep Networks via Gradient-based Localization"](https://arxiv.org/abs/1610.02391).
- **Grad-CAM++**: Chattopadhay et al., 2018. ["Grad-CAM++: Generalized Gradient-based Visual Explanations for Deep Convolutional Networks"](https://arxiv.org/abs/1710.11063).
- **XGrad-CAM**: Fu et al., 2020. ["Axiom-based Grad-CAM: Towards Accurate Visualization and Explanation of CNNs"](https://arxiv.org/abs/2008.02312).
- **HiRes-CAM**: Draelos & Carin, 2020. ["Use HiResCAM instead of Grad-CAM for faithful explanations of convolutional neural networks"](https://arxiv.org/abs/2011.08891).
- **Layer-CAM**: Jiang et al., 2021. ["LayerCAM: Exploring Hierarchical Class Activation Maps for Localization"](https://arxiv.org/abs/2104.14818).
- **Eigen-CAM / EigenGrad-CAM**: Muhammad & Yeasin, 2020. ["Eigen-CAM: Class Activation Map using Principal Components"](https://arxiv.org/abs/2008.00299).

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import cv2
import matplotlib.pyplot as plt
import os

class YOLOExplainer:
    """
    Custom Class Activation Mapping (CAM) generator for Ultralytics YOLO models.
    Supports Eigen-CAM, Grad-CAM, Grad-CAM++, XGrad-CAM, HiRes-CAM, Layer-CAM, and EigenGrad-CAM.
    """
    def __init__(self, model, target_layer, method='eigencam'):
        self.model = model
        self.target_layer = target_layer
        self.method = method.lower()
        self.activations = None
        self.gradients = None
        
        # Register hooks
        self.forward_hook = target_layer.register_forward_hook(self._forward_hook_fn)
        if self.method in ['gradcam', 'gradcam++', 'xgradcam', 'hirescam', 'layercam', 'eigengradcam']:
            self.backward_hook = target_layer.register_full_backward_hook(self._backward_hook_fn)
            
    def _forward_hook_fn(self, module, input, output):
        if isinstance(output, (list, tuple)):
            self.activations = output[0].clone()
        else:
            self.activations = output.clone()
            
    def _backward_hook_fn(self, module, grad_input, grad_output):
        if isinstance(grad_output, (list, tuple)):
            self.gradients = grad_output[0].clone()
        else:
            self.gradients = grad_output.clone()
            
    def remove_hooks(self):
        self.forward_hook.remove()
        if hasattr(self, 'backward_hook'):
            self.backward_hook.remove()
            
    def generate(self, input_tensor, class_idx=None):
        self.model.zero_grad()
        
        with torch.set_grad_enabled(True):
            if self.method in ['gradcam', 'gradcam++', 'xgradcam', 'hirescam', 'layercam', 'eigengradcam']:
                input_tensor.requires_grad = True
                
            # Ensure model weights are on the same device as input_tensor
            self.model.model.to(input_tensor.device)
            
            # Run raw PyTorch forward pass to track gradients and bypass Results wrapper
            outputs = self.model.model(input_tensor)
            
            if isinstance(outputs, (list, tuple)):
                preds = outputs[0]
            else:
                preds = outputs
                
            if self.method == 'eigencam':
                act = self.activations.detach().cpu().numpy()[0]
                C, H, W = act.shape
                matrix = act.reshape(C, H * W).T
                matrix = matrix - np.mean(matrix, axis=0)
                U, S, Vt = np.linalg.svd(matrix, full_matrices=False)
                projection = U[:, 0].reshape(H, W)
                heatmap = (projection - projection.min()) / (projection.max() - projection.min() + 1e-8)
                return heatmap
                
            elif self.method in ['gradcam', 'gradcam++', 'xgradcam', 'hirescam', 'layercam', 'eigengradcam']:
                nc = getattr(self.model.model, 'nc', 9)  # Dynamically get number of classes
                if class_idx is None:
                    class_scores = preds[0, 4:4+nc, :].sum(dim=-1)
                    class_idx = torch.argmax(class_scores).item()
                    
                score = preds[0, 4 + class_idx, :].sum()
                score.backward(retain_graph=True)
            
            act = self.activations.detach().cpu().numpy()[0]
            grad = self.gradients.detach().cpu().numpy()[0]
            
            if self.method == 'gradcam':
                weights = np.mean(grad, axis=(1, 2))
                cam = np.zeros(act.shape[1:], dtype=np.float32)
                for i, w in enumerate(weights):
                    cam += w * act[i, :, :]
                cam = np.maximum(cam, 0)
                heatmap = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
                return heatmap
                
            elif self.method == 'gradcam++':
                grads_power_2 = grad ** 2
                grads_power_3 = grad ** 3
                sum_act = np.sum(act, axis=(1, 2))
                alpha = grads_power_2 / (2 * grads_power_2 + sum_act[:, None, None] * grads_power_3 + 1e-8)
                weights = np.sum(alpha * np.maximum(grad, 0), axis=(1, 2))
                cam = np.zeros(act.shape[1:], dtype=np.float32)
                for i, w in enumerate(weights):
                    cam += w * act[i, :, :]
                cam = np.maximum(cam, 0)
                heatmap = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
                return heatmap

            elif self.method == 'xgradcam':
                sum_act = np.sum(act, axis=(1, 2))
                weights = np.sum(grad * act, axis=(1, 2)) / (sum_act + 1e-8)
                cam = np.zeros(act.shape[1:], dtype=np.float32)
                for i, w in enumerate(weights):
                    cam += w * act[i, :, :]
                cam = np.maximum(cam, 0)
                heatmap = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
                return heatmap

            elif self.method == 'hirescam':
                cam = np.sum(act * grad, axis=0)
                cam = np.maximum(cam, 0)
                heatmap = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
                return heatmap

            elif self.method == 'layercam':
                cam = np.sum(act * np.maximum(grad, 0), axis=0)
                cam = np.maximum(cam, 0)
                heatmap = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
                return heatmap

            elif self.method == 'eigengradcam':
                act_grad = act * np.maximum(grad, 0)
                C, H, W = act_grad.shape
                matrix = act_grad.reshape(C, H * W).T
                matrix = matrix - np.mean(matrix, axis=0)
                U, S, Vt = np.linalg.svd(matrix, full_matrices=False)
                projection = U[:, 0].reshape(H, W)
                heatmap = (projection - projection.min()) / (projection.max() - projection.min() + 1e-8)
                return heatmap

def show_heatmap(img_path, heatmap, title='Attention Map', save_path=None):
    img = cv2.imread(img_path)
    h, w, _ = img.shape
    heatmap_resized = cv2.resize(heatmap, (w, h))
    heatmap_color = cv2.applyColorMap(np.uint8(255 * heatmap_resized), cv2.COLORMAP_JET)
    superimposed_img = cv2.addWeighted(img, 0.6, heatmap_color, 0.4, 0)
    
    if save_path:
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        cv2.imwrite(save_path, superimposed_img)
        
    fig, axes = plt.subplots(1, 2, figsize=(12, 6))
    axes[0].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    axes[0].axis('off')
    axes[0].set_title('Original Image')
    
    axes[1].imshow(cv2.cvtColor(superimposed_img, cv2.COLOR_BGR2RGB))
    axes[1].axis('off')
    axes[1].set_title(title)
    plt.tight_layout()
    plt.show()

In [ ]:
from ultralytics import YOLO
import os
if os.path.exists('/kaggle/working/AlphaDentYolov26'):
    os.chdir('/kaggle/working/AlphaDentYolov26')

import glob
import cv2
import numpy as np
import torch

print("Starting XAI execution loop...")
test_images = glob.glob("./AlphaDent/images/valid/*.jpg") + glob.glob("./AlphaDent/images/valid/*.png")

if test_images:
    img_path = test_images[0]
    print(f"Using image for XAI: {img_path}")
    
    # Select target layer (index 22, the last C3k2 block in YOLO26 neck/backbone)
    target_layer_idx = 22
    methods = ['EigenCAM', 'GradCAM', 'GradCAM++', 'XGradCAM', 'HiResCAM', 'LayerCAM', 'EigenGradCAM']
    
    for model_name in GRID_MODELS:
        for img_size in GRID_IMAGE_SIZES:
            model_base = model_name.replace('-seg.pt', '').replace('.pt', '')
            project_name = f'yolo_seg_{model_base}_proj_{img_size}_b{TARGET_BATCH_SIZE}_e{EPOCHS}'
            xai_dir = f"runs/segment/{project_name}/xai"
            
            # Check if XAI directory already contains generated images
            all_xai_exist = True
            for method in methods:
                if not os.path.exists(os.path.join(xai_dir, f"{method}.jpg")):
                    all_xai_exist = False
                    break
                    
            if all_xai_exist and not OVERRIDE_XAI:
                print(f"Skipping XAI for {model_name} at size {img_size} (heatmaps exist in {xai_dir})")
                continue
                
            # Resolve trained weights path
            possible_paths = [
                f"runs/segment/{project_name}/train/weights/best.pt",
                f"{project_name}/train/weights/best.pt",
                f"runs/{project_name}/train/weights/best.pt",
                f"/kaggle/working/AlphaDentYolov26/runs/segment/{project_name}/train/weights/best.pt"
            ]
            weights_path = None
            for p in possible_paths:
                if os.path.exists(p):
                    weights_path = p
                    break
                    
            if not weights_path:
                print(f"⚠️ Weights not found for {model_name} at size {img_size}. Skipping XAI.")
                continue
                
            print(f"\n==================================================")
            print(f"GENERATING XAI: {model_name} | Image Size: {img_size}")
            print(f"==================================================")
            
            model_explain = YOLO(weights_path)
            target_layer = model_explain.model.model[target_layer_idx]
            
            # Prepare input tensor
            img = cv2.imread(img_path)
            img_resized = cv2.resize(img, (img_size, img_size))
            img_norm = img_resized.astype(np.float32) / 255.0
            input_tensor = torch.from_numpy(img_norm.transpose(2, 0, 1)).unsqueeze(0)
            
            val_device = DEVICE[0] if isinstance(DEVICE, list) else DEVICE
            input_tensor = input_tensor.to(val_device)
            
            for method in methods:
                save_path = os.path.join(xai_dir, f"{method}.jpg")
                if os.path.exists(save_path) and not OVERRIDE_XAI:
                    print(f"  - Skipping {method} (file exists)")
                    continue
                    
                print(f"  - Generating {method}...")
                explainer = YOLOExplainer(model_explain, target_layer, method=method)
                try:
                    heatmap = explainer.generate(input_tensor)
                    explainer.remove_hooks()
                    show_heatmap(img_path, heatmap, title=f'{model_name} {method} Map', save_path=save_path)
                except Exception as e:
                    explainer.remove_hooks()
                    print(f"  ❌ Failed to generate {method} for {model_name}: {e}")
else:
    print("No validation images found to explain.")

## 8. Export Results as ZIP

Compress all training runs, weights, validation plots, XAI heatmaps, and metrics into a single ZIP archive for easy download from Kaggle.

In [ ]:
import os
if os.path.exists('/kaggle/working/AlphaDentYolov26'):
    os.chdir('/kaggle/working/AlphaDentYolov26')

import shutil

temp_dir = "results_to_download"
runs_dir = "runs"
csv_path = "grid_search_results.csv"
zip_output = "/kaggle/working/experiment_results"

# Remove any existing temp dir
if os.path.exists(temp_dir):
    shutil.rmtree(temp_dir)
os.makedirs(temp_dir, exist_ok=True)

# Copy runs directory if it exists
if os.path.exists(runs_dir):
    print("Copying runs directory...")
    shutil.copytree(runs_dir, os.path.join(temp_dir, "runs"), dirs_exist_ok=True)

# Copy CSV results if it exists
if os.path.exists(csv_path):
    print("Copying CSV results...")
    shutil.copy2(csv_path, os.path.join(temp_dir, csv_path))

# Zip the directory
print(f"Creating ZIP archive: {zip_output}.zip...")
shutil.make_archive(zip_output, 'zip', temp_dir)

# Clean up temporary directory
shutil.rmtree(temp_dir)
print("ZIP archive created successfully! You can now download 'experiment_results.zip' from Kaggle.")